# 🛒 Shelf Void Detection — Inference Pipeline

Trained model (`best.pt`) se shelf images pe **void (khali)** aur **occupied (bhara)** detection.

## Kya karta hai ye notebook
1. Aapka trained model `best.pt` load karta hai
2. Shelf images `/content/input` me upload karte ho
3. Har image pe detect karta hai — kaunsa shelf khali hai, kaunsa bhara
4. **Color-coded annotated images** + CSV/JSON report download karta hai

## 🎨 Classes & Colors (important)
| Class | Color | Matlab |
|---|---|---|
| `missing` | 🔴 RED box | Khali shelf / void space (**unoccupied**) |
| `product` | 🟢 GREEN box | Bhara shelf (**occupied**) |

> Har box ke saath uska **confidence %** bhi likha hoga.

## Pehle kya karna hai
1. **Runtime → Change runtime type → T4 GPU** select karo
2. Apni images `/content/input` folder me upload karo (Step 3 me detail hai)
3. **Runtime → Run all** (Ctrl+F9)


## Step 1 — GPU check + Install ultralytics

In [ ]:
!nvidia-smi
import torch
print("\nGPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE — Runtime -> Change runtime type -> T4 GPU ON karo!")


In [ ]:
!pip install -q ultralytics
from ultralytics import YOLO
import os, glob, json, shutil
import cv2
import numpy as np
from pathlib import Path
from IPython.display import display, Image as IPImage
import pandas as pd
print("Ready ✅")


## Step 2 — Model (`best.pt`) load karo

Trained model `best.pt` chahiye. **2 tareeke:**

**A. Auto (sharing ke liye best 👍):** `best.pt` apne Google Drive pe upload karo →
*Anyone with the link* share karo → link me se **file ID** copy kar ke neeche `BEST_PT_DRIVE_ID` me daalo.
(ID milega: `drive.google.com/file/d/<<<YEH_ID>>>/view`)

**B. Manual:** Drive ID khali chhodo, sidebar **📁 icon → `/content` → Upload → `best.pt`** select karo.

In [ ]:
# === MODEL LOAD ===
MODEL_PATH = "/content/best.pt"
BEST_PT_DRIVE_ID = ""   # <-- yahan apne Google Drive ka file ID paste karo (brackets nahi)

if BEST_PT_DRIVE_ID.strip():
    print("Drive se best.pt download ho raha hai (ID:", BEST_PT_DRIVE_ID.strip(), ") ...")
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gdown"], check=False)
    import gdown
    gdown.download(id=BEST_PT_DRIVE_ID.strip(), output=MODEL_PATH, quiet=False)
    print()

if not os.path.exists(MODEL_PATH):
    raise FileNotFoundError(
        "best.pt nahi mila!\n"
        "Ya to BEST_PT_DRIVE_ID me Drive file ID daalo,\n"
        "ya sidebar 📁 se /content/best.pt upload karo, phir ye cell dobara run karo."
    )

model = YOLO(MODEL_PATH)
class_names = model.names

# missing (unoccupied) ka class id model se dhoondo (robust — id 0 hona chahiye)
_id_by_name = {v: k for k, v in class_names.items()}
MISSING_ID = _id_by_name.get("missing", 0)
PRODUCT_ID = _id_by_name.get("product", 1 - MISSING_ID)

print("\n✅ Model loaded :", MODEL_PATH)
print("   Classes     :", class_names)
print("   missing id  :", MISSING_ID, "(RED)   |   product id :", PRODUCT_ID, "(GREEN)")
print("   Size        : %.1f MB" % (os.path.getsize(MODEL_PATH)/1024/1024))


## Step 3 — Images upload karo

1. Sidebar **📁** kholo
2. `/content` me **`input`** naam ka folder banao (right-click → New folder)
3. Apni shelf images us `input` folder me upload karo

Neeche `IMG_DIR` path same rakho.

In [ ]:
# === CONFIGURATION — INPUT IMAGES ===
IMG_DIR = "/content/input"        # <-- images yahan upload karo
os.makedirs(IMG_DIR, exist_ok=True)

IMG_EXTENSIONS = (".jpg", ".jpeg", ".png", ".bmp", ".webp")

all_images = []
for ext in IMG_EXTENSIONS:
    all_images.extend(glob.glob(f"{IMG_DIR}/**/*{ext}", recursive=True))
    all_images.extend(glob.glob(f"{IMG_DIR}/**/*{ext.upper()}", recursive=True))
all_images = sorted(set(all_images))

print(len(all_images), "images milin (/content/input me):")
for img in all_images[:15]:
    print("  ", os.path.basename(img))
if len(all_images) > 15:
    print("  ... aur", len(all_images)-15, "more")

if len(all_images) == 0:
    print("\n⚠️  Koi image nahi mili!")
    print("   Sidebar 📁 -> /content -> 'input' folder banao -> images usme upload karo.")
    print("   Phir ye cell dobara run karo.")


## Step 4 — Settings (Confidence, IoU) + Color legend

- **CONFIDENCE** — kitna sure hona chahiye model ko (0.0 = sab dikhao, 1.0 = sirf guaranteed)
  - **0.15 (default)** → void/khali spaces zyada pakre jayein (recall ke liye)
  - 0.30–0.50 → balanced
  - 0.50–0.80 → sirf bohat confident detections
- **IoU** — overlapping boxes kitne similar hone par merge honge

Neeche wala cell **color-coded annotation helper** bhi define karta hai (🔴 missing / 🟢 product).

In [ ]:
# === TUNE THESE ===
CONFIDENCE = 0.15   # kam = zyada void detection (recall); zyada = kam false positive
IOU = 0.45          # NMS IoU threshold
IMG_SIZE = 640      # inference image size

print("Confidence :", CONFIDENCE)
print("IoU        :", IOU)
print("Image size :", IMG_SIZE)

# === COLOR-CODED ANNOTATION HELPER ===
# missing (unoccupied) -> RED, product (occupied) -> GREEN   (BGR for OpenCV)
COLOR_MISSING = (0, 0, 255)     # RED
COLOR_PRODUCT = (0, 255, 0)     # GREEN

def draw_and_count(results, src_path, out_path):
    """Color-coded boxes draw karke out_path pe save karta hai.
    Returns: (missing_n, product_n, detections_list)"""
    img = cv2.imread(src_path)
    if img is None:
        return 0, 0, []
    missing_n, product_n, dets = 0, 0, []
    for r in results:
        for box in r.boxes:
            cls = int(box.cls[0])
            conf = float(box.conf[0])
            x1, y1, x2, y2 = [int(v) for v in box.xyxy[0].tolist()]
            name = class_names[cls]
            color = COLOR_MISSING if cls == MISSING_ID else COLOR_PRODUCT

            cv2.rectangle(img, (x1, y1), (x2, y2), color, 2)
            text = name + " " + format(conf, ".2f")
            (tw, th), _ = cv2.getTextSize(text, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 1)
            ytop = max(0, y1 - th - 6)
            cv2.rectangle(img, (x1, ytop), (x1 + tw + 4, ytop + th + 6), color, -1)
            cv2.putText(img, text, (x1 + 2, ytop + th + 2),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1, cv2.LINE_AA)
            if cls == MISSING_ID:
                missing_n += 1
            else:
                product_n += 1
            dets.append({"class": name, "confidence": round(conf, 4),
                         "bbox": [round(v, 1) for v in (x1, y1, x2, y2)]})
    cv2.imwrite(out_path, img)
    return missing_n, product_n, dets

print("✅ Color-coded annotation ready: RED=missing (void), GREEN=product (occupied)")


## Step 5 — Single Image Inference (pehla test)

Pehli image pe test karo — boxes sahi aa rahe hain ya nahi.

In [ ]:
assert len(all_images) > 0, "Pehle /content/input me images upload karo (Step 3)."

test_img = all_images[0]
print("Testing:", os.path.basename(test_img), "\n")

results = model.predict(source=test_img, conf=CONFIDENCE, iou=IOU,
                        imgsz=IMG_SIZE, save=False, verbose=False)

os.makedirs("/content/single_test", exist_ok=True)
out_path = "/content/single_test/" + os.path.basename(test_img)
m, p, dets = draw_and_count(results, test_img, out_path)

print("Detections:", len(dets), " -> ", p, "product (green),", m, "missing (red)")
for d in dets[:12]:
    print("  %-8s conf=%.3f  bbox=%s" % (d["class"], d["confidence"], d["bbox"]))
if len(dets) > 12:
    print("  ... aur", len(dets)-12)

display(IPImage(filename=out_path, width=800))


## Step 6 — Batch Inference (sari images)

Ab saari images pe inference chalao. Har image ka color-coded version + CSV row banega.

In [ ]:
OUTPUT_DIR = "/content/inference_results"
ANNOTATED_DIR = OUTPUT_DIR + "/annotated"
os.makedirs(ANNOTATED_DIR, exist_ok=True)

csv_rows = []
print("Processing", len(all_images), "images...\n")

for i, img_path in enumerate(all_images):
    fname = os.path.basename(img_path)
    results = model.predict(source=img_path, conf=CONFIDENCE, iou=IOU,
                            imgsz=IMG_SIZE, save=False, verbose=False)
    out_path = os.path.join(ANNOTATED_DIR, fname)
    m, p, dets = draw_and_count(results, img_path, out_path)

    total = m + p
    fill_pct = round(100.0 * p / total, 1) if total > 0 else None

    csv_rows.append({
        "image": fname,
        "missing_count": m,
        "product_count": p,
        "total_detections": total,
        "shelf_fill_pct": fill_pct,
        "detections_json": json.dumps(dets),
    })

    flag = "⚠️ VOID" if m > 0 else "ok"
    print("  [%d/%d] %s: %d product, %d missing  -> fill %s%%  %s" %
          (i+1, len(all_images), fname, p, m, fill_pct, flag))

print("\n✅ Done -", len(all_images), "images processed ->", ANNOTATED_DIR)


## Step 7 — Results Summary Table

In [ ]:
df = pd.DataFrame(csv_rows)
df_display = df.drop(columns=["detections_json"])

print("=== DETECTION SUMMARY ===")
print("Total images            :", len(df))
print("Images with void/missing:", int((df["missing_count"] > 0).sum()))
print("Images fully occupied   :", int((df["missing_count"] == 0).sum()))
print("\nTotal missing (RED)   :", int(df["missing_count"].sum()))
print("Total product (GREEN) :", int(df["product_count"].sum()))
if len(df):
    print("Avg shelf fill %      : %.1f%%" % df["shelf_fill_pct"].mean())
print()
display(df_display)


## Step 8 — CSV Report Download

In [ ]:
csv_path = OUTPUT_DIR + "/inference_report.csv"
df.to_csv(csv_path, index=False)
print("CSV saved:", csv_path)

from google.colab import files
files.download(csv_path)
print("CSV download ho raha hai...")


## Step 9 — Annotated Images ZIP Download

Saari color-coded images ka zip.

In [ ]:
annotated = sorted(glob.glob(ANNOTATED_DIR + "/*"))
print(len(annotated), "annotated images")

zip_base = "/content/annotated_results"
shutil.make_archive(zip_base, "zip", ANNOTATED_DIR)
print("Zip size: %.1f MB" % (os.path.getsize(zip_base + ".zip")/1024/1024))

files.download(zip_base + ".zip")
print("Annotated images zip download ho raha hai...")


## Step 10 — Sample Results Gallery

Ek void wali + ek fully-occupied wali image dikhao.

In [ ]:
shown = set()
for row in csv_rows:
    category = "void" if row["missing_count"] > 0 else "occupied"
    if category in shown:
        continue
    path = os.path.join(ANNOTATED_DIR, row["image"])
    if not os.path.exists(path):
        continue
    if category == "void":
        print("\n--- VOID DETECTED:", row["image"], " (", row["missing_count"], "missing,", row["product_count"], "product) ---")
    else:
        print("\n--- FULLY OCCUPIED:", row["image"], " (0 missing,", row["product_count"], "product) ---")
    display(IPImage(filename=path, width=700))
    shown.add(category)
    if len(shown) >= 2:
        break


## Step 11 — Raw Predictions JSON (developer ke liye)

App/demo banana ho toh raw JSON yahan se mil jayega.

In [ ]:
json_path = OUTPUT_DIR + "/predictions.json"
with open(json_path, "w") as f:
    json.dump(csv_rows, f, indent=2)
print("Predictions JSON:", json_path)
print("Size: %.1f KB" % (os.path.getsize(json_path)/1024))

files.download(json_path)


---

## 📦 Output Files Summary

| File | Description |
|---|---|
| `inference_report.csv` | Har image ka void/product count + **shelf fill %** + full detection details |
| `annotated_results.zip` | Color-coded images (🔴 missing / 🟢 product + confidence) |
| `predictions.json` | Raw detection data (app/demo integration ke liye) |

## 🎨 Color Legend
- 🔴 **RED box** = `missing` (khali shelf / void → restock karo)
- 🟢 **GREEN box** = `product` (occupied)

## Masla aaye toh
- **"best.pt not found"** → `BEST_PT_DRIVE_ID` daalo ya sidebar 📁 se upload karo
- **"GPU not available"** → Runtime → Change runtime type → T4 GPU
- **"0 images milin"** → images `/content/input` folder me upload karo (Step 3)
- **Kam detections** → CONFIDENCE kam karo (0.10–0.15)
- **Zyada false positives** → CONFIDENCE badhao (0.40–0.50)
- **Colab disconnect** → dobara Run All karo, results regenerate ho jayenge

---
*Inference Pipeline • YOLOv11 • Custom color-coded annotation • Google Colab*